In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import numpy as np
import subprocess
import io
from scipy.io import wavfile

def extract_audio_in_memory(video_path):
    """FFmpeg를 사용해 영상에서 오디오를 추출하고 파일 저장 없이 메모리(BytesIO)로 반환합니다."""
    command = [
        'ffmpeg',
        '-i', video_path,
        '-f', 'wav',          # WAV 포맷 지정
        '-ac', '1',           # 모노(1채널) 채널로 변환하여 연산 단순화
        '-ar', '44100',       # 샘플링 레이트 44.1kHz
        '-loglevel', 'quiet', # 콘솔 출력 생략
        'pipe:1'              # 출력을 파이프(stdout)로 보냄
    ]

    # FFmpeg 프로세스 실행 및 메모리 캡처
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    out, err = process.communicate()

    if process.returncode != 0:
        raise RuntimeError(f"FFmpeg 오디오 추출 실패: {err.decode('utf-8')}")

    # 메모리 버퍼(out)에 담긴 WAV 데이터를 scipy로 읽기
    sample_rate, audio_data = wavfile.read(io.BytesIO(out))
    return sample_rate, audio_data

def calculate_rms(audio_array):
    """오디오 배열의 RMS(Root Mean Square)를 계산합니다."""
    if audio_array is None or len(audio_array) == 0:
        return 0.0
    # 오버플로우 방지를 위해 float64로 변환 후 연산
    audio_array = audio_array.astype(np.float64)
    return np.sqrt(np.mean(np.square(audio_array)))

def add_volume_to_json_in_memory(video_path, json_path, output_json_path):
    # 1. JSON 파일 로드
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 2. 영상에서 오디오 데이터를 메모리로 즉시 추출
    print("비디오에서 오디오 데이터를 메모리로 스트리밍 중 (WAV 포맷)...")
    sample_rate, audio_data = extract_audio_in_memory(video_path)

    rms_values = []

    # 3. 각 세그먼트별 오디오 볼륨(RMS) 분석
    print("세그먼트 구간별 볼륨 연산 중...")
    for segment in data:
        # 초(second) 단위의 시간을 배열 인덱스로 변환
        start_index = int(segment['start'] * sample_rate)
        end_index = int(segment['end'] * sample_rate)

        # 배열 슬라이싱 (메모리 접근이므로 속도가 매우 빠름)
        segment_audio = audio_data[start_index:end_index]

        rms = calculate_rms(segment_audio)
        rms_values.append(rms)
        segment['raw_volume'] = rms  # 연산을 위해 임시 저장

    # 4. 0 ~ 1 사이로 정규화 (Min-Max Scaling)
    min_rms = np.min(rms_values)
    max_rms = np.max(rms_values)

    for segment in data:
        raw_vol = segment.pop('raw_volume')

        if max_rms - min_rms == 0:
            normalized_vol = 0.0
        else:
            normalized_vol = (raw_vol - min_rms) / (max_rms - min_rms)

        segment['normalized_volume'] = round(normalized_vol, 4)

    # 5. 결과 저장
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"완료! 정규화된 볼륨 데이터가 '{output_json_path}'에 저장되었습니다.")

In [ ]:
VIDEO_FILE = "your_video_file.mp4"       # 분석할 영상
INPUT_JSON = "source4_data.json"         # 원본 JSON 파일
OUTPUT_JSON = "source4_with_volume.json" # 최종 저장될 JSON 파일

import glob

folder_path = "/content/drive/MyDrive/stt/downloads/**"
mp3_list = sorted(glob.glob(folder_path + "/*.mp3", recursive=True))
json_list = sorted(glob.glob(folder_path + "/*.json", recursive=True))
for mp3_path in mp3_list:
  add_volume_to_json_in_memory(VIDEO_FILE, INPUT_JSON, OUTPUT_JSON)